# 🌦️ Weather RAG Monitor Demo

This notebook follows the same notebook-first RAG learning style: load data → normalize → create documents → split → embed → store in Chroma → retrieve → generate with Gemini.

In [1]:
from weather_rag.scraper import OpenMeteoScraper
from weather_rag.rag import weather_documents, build_vector_store, answer_question


In [2]:
scraper = OpenMeteoScraper()
geo = scraper.geocode('Bettiah', 'India')
geo


{'id': 1276393,
 'name': 'Bettiah',
 'latitude': 26.80229,
 'longitude': 84.50311,
 'elevation': 84.0,
 'feature_code': 'PPL',
 'country_code': 'IN',
 'admin1_id': 1275715,
 'admin2_id': 1260208,
 'admin3_id': 12685602,
 'timezone': 'Asia/Kolkata',
 'population': 132209,
 'country_id': 1269750,
 'country': 'India',
 'admin1': 'Bihar',
 'admin2': 'Pashchim Champaran',
 'admin3': 'Bettiah'}

In [3]:
forecast = scraper.fetch_forecast(
    latitude=geo['latitude'],
    longitude=geo['longitude'],
    timezone=geo.get('timezone', 'auto'),
    forecast_days=7,
)
documents = weather_documents(
    forecast,
    geo['name'],
    geo['latitude'],
    geo['longitude'],
)
len(documents), documents[0]


(168,
 Document(metadata={'source': 'Open-Meteo', 'location': 'Bettiah', 'timestamp': '2026-09-01T00:00', 'latitude': 26.80229, 'longitude': 84.50311}, page_content='Weather forecast for Bettiah\nCoordinates: 26.80229, 84.50311\nLocal time: 2026-09-01T00:00\nTemperature: 27.6 °C\nApparent temperature: 34.5 °C\nRelative humidity: 96 %\nPrecipitation: 0.1 mm\nRain: 0.0 mm\nWeather condition: Light drizzle\nCloud cover: 100 %\nWind speed: 7.2 km/h\nWind direction: 55 degrees\nSource: Open-Meteo'))

In [4]:
vector_store = build_vector_store(documents)
docs = vector_store.similarity_search('Will it rain tomorrow?', k=5)
for doc in docs:
    print(doc.page_content, '\n---')


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-2\nPlease retry in 14.187699363s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-e

In [ ]:
print(answer_question('Will it rain tomorrow?', vector_store, 'Bettiah'))
